# Problem Set 4, Problem 4: DDPM and DDIM sampling
ROB-GY 6013 Mathematics for Robotics, Fall 2026

**Instructions.** Write code only where a cell says `TODO`, and run all cells from top to bottom. Use only NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## The model

The demonstrated lateral position $X_0$ of a robot passing an obstacle comes from two sides: with probability $\tfrac23$ it is near $-1$ (left) and with probability $\tfrac13$ near $3$ (right), each with standard deviation $0.2$:
$$p_0(x)=\tfrac23\,\mathcal N(x;-1,0.2^2)+\tfrac13\,\mathcal N(x;3,0.2^2).$$

The forward process uses $K=100$ levels with $\beta_k$ spaced evenly from $10^{-4}$ to $0.2$, so $a_k=1-\beta_k$, $\bar a_k=a_1\cdots a_k$, $\alpha_k=\sqrt{\bar a_k}$ and $\sigma_k=\sqrt{1-\bar a_k}$. The arrays below are indexed by $k=0,1,\dots,K$, with $\bar a_0=1$.

At level $k$ the noisy position $X_k=\alpha_kX_0+\sigma_k\epsilon$ is again a mixture of two Gaussians, so its score $s_k(x)$ is known exactly. Instead of a trained network, `eps_hat(x, k)` returns the exact noise prediction $\hat\epsilon=-\sigma_k s_k(x)$, as in Problem 3(e). Nothing to change here.

In [ ]:
K = 100
beta = np.concatenate([[0.0], np.linspace(1e-4, 0.2, K)])   # beta[k] for k = 1..K
a = 1 - beta
abar = np.cumprod(a)                                         # abar[0] = 1
alpha = np.sqrt(abar)
sigma = np.sqrt(1 - abar)

w = np.array([2 / 3, 1 / 3])     # weights of the two sides
m = np.array([-1.0, 3.0])        # their mean positions
s0 = 0.2                         # their standard deviation

def p0(x):
    """Density of the demonstrated positions."""
    return sum(wj * np.exp(-(x - mj) ** 2 / (2 * s0 ** 2)) / np.sqrt(2 * np.pi * s0 ** 2) for wj, mj in zip(w, m))

def eps_hat(x, k):
    """Exact noise prediction -sigma_k s_k(x) at level k >= 1, for an array x."""
    mu = alpha[k] * m                          # component means at level k
    var = alpha[k] ** 2 * s0 ** 2 + sigma[k] ** 2
    logp = np.log(w) - (x[:, None] - mu) ** 2 / (2 * var)
    r = np.exp(logp - logp.max(axis=1, keepdims=True))
    r /= r.sum(axis=1, keepdims=True)          # responsibilities
    score = (r * (-(x[:, None] - mu) / var)).sum(axis=1)
    return -sigma[k] * score

print(f"alpha_K = {alpha[K]:.4f}, sigma_K = {sigma[K]:.4f}")

## Part (a): one DDPM step

Complete one reverse step from level $k$ to level $k-1$: draw $x_{k-1}\sim\mathcal N(\mu,\tilde\beta_k)$, that is, $x_{k-1}=\mu+\sqrt{\tilde\beta_k}\,z$, with

- mean $\mu=\dfrac{1}{\sqrt{a_k}}\Bigl(x_k-\dfrac{\beta_k}{\sigma_k}\hat\epsilon\Bigr)$, where $\hat\epsilon$ is `eps_hat(x, k)`;
- variance $\tilde\beta_k=\beta_k\dfrac{1-\bar a_{k-1}}{1-\bar a_k}$.

For $k=1$, return $\mu$ with no fresh noise.

With the noise `z` passed in, the test should print `[ 0.80181427 -0.79436148]` and then `[-1.00004988  3.00014964]`.

In [ ]:
def ddpm_step(x, k, z):
    """One DDPM step from level k to level k - 1, with fresh noise z."""
    # TODO (a): compute the mean, and add sqrt(beta_tilde) * z unless k == 1.
    raise NotImplementedError("part (a): DDPM step")

# Test
print(ddpm_step(np.array([0.5, -0.5]), 50, np.array([1.0, -1.0])))
print(ddpm_step(np.array([-1.0, 3.0]), 1, np.array([1.0, -1.0])))

### Sampling 2000 positions

The next cell starts $2000$ samples from $x_K\sim\mathcal N(0,1)$, applies `ddpm_step` for $k=K,\dots,1$, and saves a histogram of the samples, the density $p_0$, and the mean $E[X_0]$, as `p4-ddpm.png`. For part (b) it prints the fraction of samples on each side of $x=1$ and the Monte Carlo standard error $\sqrt{\hat p(1-\hat p)/N}$ of a fraction $\hat p$. Nothing to change here.

In [ ]:
N = 2000
rng = np.random.default_rng(0)
x = rng.standard_normal(N)
for k in range(K, 0, -1):
    x = ddpm_step(x, k, rng.standard_normal(N))
samples = x

mean_action = w @ m
grid = np.linspace(-2.5, 4.5, 400)
plt.figure(figsize=(5, 3.5))
plt.hist(samples, bins=70, range=(-2.5, 4.5), density=True, alpha=0.5, label="DDPM samples")
plt.plot(grid, p0(grid), "k-", label="demonstrations $p_0$")
plt.axvline(mean_action, color="C3", ls="--", label="mean $E[X_0]$")
plt.xlabel("lateral position")
plt.ylabel("density")
plt.legend()
plt.title("DDPM with K = 100 steps")
plt.savefig("p4-ddpm.png", dpi=150, bbox_inches="tight")
plt.show()

for name, frac in [("left  (x < 1)", np.mean(samples < 1)), ("right (x > 1)", np.mean(samples > 1))]:
    print(f"{name}: fraction {frac:.3f}, standard error {np.sqrt(frac * (1 - frac) / N):.3f}")
print(f"mean E[X0] = {mean_action:.4f}")

## Part (c): one DDIM step

Complete one deterministic DDIM step from level $k$ to a lower level $j$:

- the estimate $\hat x_0=\dfrac{x_k-\sigma_k\hat\epsilon}{\alpha_k}$ with $\hat\epsilon$ from `eps_hat(x, k)`;
- the new sample $x_j=\alpha_j\hat x_0+\sigma_j\hat\epsilon$.

The test should print `[ 0.5746785  -0.53051656]` and then `[-1.01382785  3.04148355]`.

In [ ]:
def ddim_step(x, k, j):
    """One DDIM step from level k to level j < k."""
    # TODO (c): estimate x0 from x and eps_hat(x, k), then return the sample at level j.
    raise NotImplementedError("part (c): DDIM step")

# Test
print(ddim_step(np.array([0.5, -0.5]), 50, 40))
print(ddim_step(np.array([-1.0, 3.0]), 10, 0))

### DDIM with fewer steps

The next cell runs `ddim_step` on evenly spaced levels from $K$ down to $0$, with $100$, $10$, $2$ and $1$ steps from the same starting noise. Each step calls `eps_hat` once. It prints the fraction of samples on the left ($x<0$), at the obstacle ($0<x<2$) and on the right ($x>2$), and saves the histograms as `p4-ddim.png`. Nothing to change here.

In [ ]:
def ddim_sample(n_steps, N, rng):
    """DDIM samples with n_steps calls of eps_hat, from x_K ~ N(0, 1)."""
    ks = np.round(np.linspace(K, 0, n_steps + 1)).astype(int)
    x = rng.standard_normal(N)
    for k, j in zip(ks[:-1], ks[1:]):
        x = ddim_step(x, k, j)
    return x

fig, axes = plt.subplots(1, 4, figsize=(12, 2.8), sharey=True)
print(f"{'steps':>5}  {'left':>5}  {'obstacle':>8}  {'right':>5}")
for ax, n_steps in zip(axes, [100, 10, 2, 1]):
    x = ddim_sample(n_steps, N, np.random.default_rng(1))
    print(f"{n_steps:5d}  {np.mean(x < 0):5.3f}  {np.mean((x > 0) & (x < 2)):8.3f}  {np.mean(x > 2):5.3f}")
    ax.hist(x, bins=70, range=(-2.5, 4.5), density=True, color="C0")
    ax.plot(grid, p0(grid), "k-")
    ax.axvline(mean_action, color="C3", ls="--", zorder=0)
    ax.set_title(f"DDIM, {n_steps} step{'s' if n_steps > 1 else ''}")
    ax.set_xlabel("lateral position")
axes[0].set_ylabel("density")
axes[0].set_ylim(0, 2.2)          # a taller bar is cut off at the top
plt.savefig("p4-ddim.png", dpi=150, bbox_inches="tight")
plt.show()